In [ ]:
import os, glob, time
import numpy as np, pandas as pd
from joblib import Parallel, delayed
def find_root():
    for r in ["/kaggle/input/rogii-wellbore-geology-prediction",
              "/kaggle/input/competitions/rogii-wellbore-geology-prediction","data"]+sorted(glob.glob("/kaggle/input/*")):
        if glob.glob(os.path.join(r,"train","*__horizontal_well.csv")): return r
    raise FileNotFoundError("data")
DATA=find_root(); print("DATA:",DATA,flush=True)
MOM=0.998
def pf(md,z,gr,dgr,tw_tvt,tw_gr,tw_dgr,ls,ir,gs,gsg,N,seed,GW):
    rng=np.random.default_rng(seed)
    pos=ls+2.0*rng.standard_normal(N); rate=ir+0.01*rng.standard_normal(N); w=np.ones(N)/N
    lo,hi=tw_tvt[0]-60,tw_tvt[-1]+60; out=np.empty(len(md)); pm=md[0]-1
    for i in range(len(md)):
        dm=max(md[i]-pm,1.0); rate=MOM*rate+0.002*rng.standard_normal(N); pos=pos+rate*dm+0.005*rng.standard_normal(N)
        tvt=np.clip(pos-z[i],lo,hi); pos=tvt+z[i]
        if np.isfinite(gr[i]):
            d=(gr[i]-np.interp(tvt,tw_tvt,tw_gr))/gs
            lk=np.maximum(np.exp(-0.5*np.minimum(d*d,600)),1e-300)
            if np.isfinite(dgr[i]):
                dd=(dgr[i]-np.interp(tvt,tw_tvt,tw_dgr))/gsg
                lg=np.maximum(np.exp(-0.5*np.minimum(dd*dd,600)),1e-300)
                lk=(1-GW)*lk+GW*lg
            w*=lk; s=w.sum(); w=w/s if s>0 else np.ones(N)/N
        if 1.0/(w*w).sum()<0.5*N:
            cum=np.cumsum(w); u0=rng.uniform(0,1.0/N)
            idx=np.clip(np.searchsorted(cum,u0+np.arange(N)/N),0,N-1)
            pos=pos[idx]+0.1*rng.standard_normal(N); rate=rate[idx]+0.001*rng.standard_normal(N); w=np.ones(N)/N
        out[i]=float(np.dot(w,pos-z[i])); pm=md[i]
    return out
def predict(w,GW,NS=16,N=300):
    h=pd.read_csv(f"{DATA}/train/{w}__horizontal_well.csv",usecols=["MD","Z","GR","TVT","TVT_input"])
    tw=pd.read_csv(f"{DATA}/train/{w}__typewell.csv").dropna(subset=["TVT","GR"]).sort_values("TVT")
    ps=int(h["TVT_input"].notna().sum())
    if ps<60 or ps>=len(h)-5 or len(tw)<10: return None
    tt=tw["TVT"].to_numpy(float); tg=tw["GR"].to_numpy(float); tdg=np.gradient(pd.Series(tg).rolling(7,center=True,min_periods=1).mean().to_numpy())
    kn=h.iloc[:ps]; tps=float(h["TVT"].iloc[ps-1])
    at=np.interp(kn["TVT_input"].to_numpy(),tt,tg)
    gs=float(np.clip(np.nanstd(kn["GR"].fillna(0).to_numpy()-at),10.,60.))
    tail=kn.tail(30); dt=np.diff(tail["TVT_input"].to_numpy()); dz=np.diff(tail["Z"].to_numpy()); dmv=np.diff(tail["MD"].to_numpy()); m=dmv>0
    ir=float(np.median((dt+dz)[m]/dmv[m])) if m.sum()>=3 else 0.
    gr=h["GR"].interpolate(limit_direction="both").fillna(tg.mean()).to_numpy()
    dgr=np.gradient(pd.Series(gr).rolling(7,center=True,min_periods=1).mean().to_numpy())
    gsg=max(np.nanstd(np.diff(kn["GR"].fillna(tg.mean()).to_numpy())),0.5)
    ev=np.arange(ps,len(h)); idx=ev
    md=np.concatenate([[float(kn["MD"].iloc[-1])],h["MD"].to_numpy(float)[idx]])
    z=np.concatenate([[float(kn["Z"].iloc[-1])],h["Z"].to_numpy(float)[idx]])
    grv=np.concatenate([[np.nan],gr[idx]]); dgrv=np.concatenate([[np.nan],dgr[idx]])
    ls=tps+float(kn["Z"].iloc[-1])
    P=np.empty((NS,len(md)))
    for s in range(NS): P[s]=pf(md,z,grv,dgrv,tt,tg,tdg,ls,ir,gs,gsg,N,s,GW)
    est=P.mean(0)[1:]
    y=h["TVT"].to_numpy(float)[idx]
    return y,est,np.full(len(y),tps)
wells=[os.path.basename(f).split("__")[0] for f in sorted(glob.glob(f"{DATA}/train/*__horizontal_well.csv"))]
print("wells",len(wells),flush=True); t0=time.time()
for GW in [0.0,0.3,0.5]:
    res=[r for r in Parallel(n_jobs=4,verbose=5)(delayed(predict)(w,GW) for w in wells) if r is not None]
    def pooled(k): 
        e=[(y-(p if k=="pf" else c))**2 for y,p,c in res]; return float(np.sqrt(np.mean(np.concatenate(e))))
    print(f"GW={GW}  const {pooled('c'):.3f}  shape-PF {pooled('pf'):.3f}  ({time.time()-t0:.0f}s, n={len(res)})",flush=True)
print("DONE",flush=True)